# Section 2: Data Preparation

---

This section walks through the steps we used to clean and standardize the monster bestiary. The goal is to turn the raw scraped text into a consistent, tabular dataset that is easy to analyze and easy to learn from.

### What does “good” data look like?
Here are the qualities we're checking for as we clean the dataset:
* **Accuracy:** Does each row reflect the monster stats correctly?
* **Completeness:** Are important columns (e.g., Ability Scores, CR) filled in?
* **Timeliness:** Is the data aligned with 5th Edition rules, not older/incorrect versions?
* **Consistency:** Do values use uniform formats (e.g., HP as a number, AC as a number)?
* **Uniqueness:** Are there duplicate monster entries that need consolidation?
* **Usefulness:** Is the data detailed enough to support the analysis without extra clutter?

If the dataset is messy, the analysis will be noisy and models can learn the wrong patterns.

## 1. First look at the data
Before we start changing things, let’s load the raw CSV and take a quick peek at what we’re working with: how big the file is, what columns exist, and what the first few rows look like.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('../data/raw_bestiary.csv')

# View the total count of monsters and stats.
print("Original shape of data:", df.shape)

Original shape of data: (4454, 34)


In [2]:
# Let's see the first five rows of the dataset
display(df.head())

,Name,Source,Page,Size,Type,Alignment,AC,HP,Speed,Strength,...,Traits,Actions,Bonus Actions,Reactions,Legendary Actions,Mythic Actions,Lair Actions,Regional Effects,Environment,Treasure
0,"""The Demogorgon""",IMR,53,Large,Giant,chaotic neutral,15 (natural armor),123 (13d12 + 39),40 ft.,21,...,Two Heads. The ettin has advantage on Wisdom (...,Multiattack. The ettin makes two attacks: one ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aarakocra,MM'14,12,Medium,Humanoid (Aarakocra),neutral good,12,13 (3d8),"20 ft., Fly 50 ft.",10,...,Dive Attack. If the aarakocra is flying and di...,"Talon. Melee Weapon Attack: +4 to hit, reach 5...",NaN,NaN,NaN,NaN,NaN,NaN,Mountain,NaN
2,Aarakocra Aeromancer,MM'25,10,Medium,Elemental,neutral,16,66 (12d8 + 12),"20 ft., Fly 50 ft.",10,...,NaN,Multiattack. The aarakocra makes two Wind Staf...,NaN,Feather Fall (1/Day). The aarakocra casts Feat...,NaN,NaN,NaN,NaN,"Mountain, Planar (Elemental Plane of Air)","Individual, Implements"
3,Aarakocra Simulacrum,SKT,188,Medium,Humanoid (Aarakocra),neutral good,12,6 (3d4),"20 ft., Fly 50 ft.",10,...,Dive Attack. If the aarakocra is flying and di...,"Talon. Melee Weapon Attack: +4 to hit, reach 5...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aarakocra Skirmisher,MM'25,10,Medium,Elemental,neutral,12,11 (2d8 + 2),"20 ft., Fly 50 ft.",10,...,NaN,"Talons. Melee Attack Roll: +4, reach 5 ft. Hit...",NaN,NaN,NaN,NaN,NaN,NaN,"Mountain, Planar (Elemental Plane of Air)","Individual, Implements"


In [3]:
# let's check the column names and datatypes
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4454 entries, 0 to 4453
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Name                    4454 non-null   str  
 1   Source                  4454 non-null   str  
 2   Page                    4454 non-null   str  
 3   Size                    4454 non-null   str  
 4   Type                    4454 non-null   str  
 5   Alignment               4375 non-null   str  
 6   AC                      4449 non-null   str  
 7   HP                      4449 non-null   str  
 8   Speed                   4454 non-null   str  
 9   Strength                4454 non-null   str  
 10  Dexterity               4454 non-null   str  
 11  Constitution            4454 non-null   str  
 12  Intelligence            4454 non-null   str  
 13  Wisdom                  4454 non-null   str  
 14  Charisma                4454 non-null   str  
 15  Saving Throws           1989 non

In [4]:
# Let's get a statistical summary
df.describe()

,Name,Source,Page,Size,Type,Alignment,AC,HP,Speed,Strength,...,Traits,Actions,Bonus Actions,Reactions,Legendary Actions,Mythic Actions,Lair Actions,Regional Effects,Environment,Treasure
count,4454,4454,4454,4454,4454,4375,4449,4449,4454,4454,...,3753,4414,582,571,432,17,159,200,1387,343
unique,3733,105,374,9,238,26,260,810,313,31,...,3256,3767,520,518,377,13,122,139,201,11
top,Expert,MM'25,undefined,Medium,Monstrosity,unaligned,12,11 (2d8 + 2),30 ft.,10,...,Legendary Resistance (3/Day). If the dragon fa...,"Club. Melee Weapon Attack: +2 to hit, reach 5 ...",Change Shape. The dragon magically transforms ...,Rock Catching. If a rock or similar object is ...,Detect. The dragon makes a Wisdom (Perception)...,Bite. The greatwyrm makes one Bite attack.\r\n...,"As they are presented in the Monster Manual, f...",The region containing a faerie dragon's lair c...,Underdark,Any
freq,4,503,99,2250,381,871,359,88,1689,470,...,20,55,10,8,7,5,5,5,170,88


## 2. Data Cleaning 

### Handling missing values
We’ll look for empty or placeholder values (like "None" or "Unknown"), decide what makes sense to keep, and replace blanks with more consistent defaults.

In [5]:
# let's first see what are the unique values that the CR column has
unique_crs = df['CR'].unique()
print(unique_crs)

<StringArray>
[                                     '8 (XP 3,900; PB +3)',
                                       '1/4 (XP 50; PB +2)',
                                      '4 (XP 1,100; PB +2)',
                                       '1/8 (XP 25; PB +2)',
                                      '6 (XP 2,300; PB +3)',
                                        '3 (XP 700; PB +2)',
                                        '2 (XP 450; PB +2)',
                                              'None (XP 0)',
            'None (XP 0; PB equals your Proficiency Bonus)',
                                      '9 (XP 5,000; PB +4)',
                                    '22 (XP 41,000; PB +7)',
                                     '10 (XP 5,900; PB +4)',
                   '10 (XP 5,900, or 7,200 in lair; PB +4)',
                                      '5 (XP 1,800; PB +3)',
                                    '23 (XP 50,000; PB +7)',
                 '16 (XP 15,000, or 18,000 in lair; PB +5)',
          

After looking at all the possible values, let's get rid of the values with none or unkonwn to them as we are not intrested on evaluating them

In [6]:
# Count rows before filtering
original_count = len(df)

# Remove rows with invalid CR
df = df[~df['CR'].str.startswith(('None', 'Unknown'), na=False)].copy()

# Print results
print(f"Removed {original_count - len(df)} rows with invalid CR.")
print(f"New shape: {df.shape}")

Removed 102 rows with invalid CR.
New shape: (4352, 34)


Now, let's handle the missing values of the other features

In [7]:
# Count NaNs and empty or whitespace strings
def count_all_blanks(series):
    nan_count = series.isnull().sum()
    empty_str_mask = series.apply(lambda x: str(x).strip() == "" if pd.notnull(x) else False)
    return nan_count + empty_str_mask.sum()

# Define fill strategy by column name
def get_fill_strategy(col):
    if col == 'Alignment': return 'unaligned'
    if col == 'Environment': return 'Unknown'
    return 'None'

# Generate missing value report
blank_data = []
for col in df.columns:
    total_blanks = count_all_blanks(df[col])
    if total_blanks > 0:
        blank_data.append({
            'Column': col,
            'Blank Count': total_blanks,
            'Fill Strategy': get_fill_strategy(col),
            'Percentage (%)': round((total_blanks / len(df)) * 100, 2)
        })

blank_summary_df = pd.DataFrame(blank_data).sort_values(by='Blank Count', ascending=False)

print("DATA AUDIT: MISSING VALUES")
print("="*70)
print(blank_summary_df.to_string(index=False))
print("="*70)

# List columns for 'None' fill
none_fill_cols = [
    'Saving Throws', 'Skills', 'Damage Vulnerabilities', 'Damage Resistances', 
    'Damage Immunities', 'Condition Immunities', 'Traits', 'Actions', 
    'Bonus Actions', 'Reactions', 'Legendary Actions', 'Mythic Actions', 
    'Lair Actions', 'Regional Effects', 'Treasure', 'Languages'
]

# Fill missing values
df['Environment'] = df['Environment'].fillna("Unknown")
df['Alignment'] = df['Alignment'].fillna("unaligned")
df[none_fill_cols] = df[none_fill_cols].fillna("None")

# Print summary
print(f"\nSuccess:")
print(f"- Filled {len(none_fill_cols)} columns with 'None'")
print(f"- Filled 'Environment' with 'Unknown'")
print(f"- Filled 'Alignment' with 'unaligned'")

DATA AUDIT: MISSING VALUES
                Column  Blank Count Fill Strategy  Percentage (%)
        Mythic Actions         4335          None           99.61
          Lair Actions         4193          None           96.35
Damage Vulnerabilities         4191          None           96.30
      Regional Effects         4152          None           95.40
              Treasure         4009          None           92.12
     Legendary Actions         3920          None           90.07
             Reactions         3798          None           87.27
         Bonus Actions         3781          None           86.88
    Damage Resistances         2986          None           68.61
           Environment         2965       Unknown           68.13
     Damage Immunities         2822          None           64.84
  Condition Immunities         2693          None           61.88
         Saving Throws         2394          None           55.01
                Skills         1225          None

To make the dataset complete, we decided to fill every missing spaces with the appropriate value (None, Unknown, unaligned)

In [8]:
#let's verify
df.info()

<class 'pandas.DataFrame'>
Index: 4352 entries, 0 to 4453
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Name                    4352 non-null   str  
 1   Source                  4352 non-null   str  
 2   Page                    4352 non-null   str  
 3   Size                    4352 non-null   str  
 4   Type                    4352 non-null   str  
 5   Alignment               4352 non-null   str  
 6   AC                      4352 non-null   str  
 7   HP                      4352 non-null   str  
 8   Speed                   4352 non-null   str  
 9   Strength                4352 non-null   str  
 10  Dexterity               4352 non-null   str  
 11  Constitution            4352 non-null   str  
 12  Intelligence            4352 non-null   str  
 13  Wisdom                  4352 non-null   str  
 14  Charisma                4352 non-null   str  
 15  Saving Throws           4352 non-null

from here, we can see that every row of every column is now filled

### 2.3 Handling Duplicates
If a monster appears twice (e.g., in the *Monster Manual* and *Mordenkainen Presents: Monsters of the Multiverse*), we will keep the most recent version based on the source hierarchy and remove the outdated entry.

In [9]:
# Find names that appear more than once
duplicate_mask = df.duplicated(subset=['Name'], keep=False)
duplicates_df = df[duplicate_mask][['Name', 'Source']].sort_values(by='Name')

print(f"Total duplicate rows found: {len(duplicates_df)}")
print("-" * 30)
print(duplicates_df.head(100))

Total duplicate rows found: 1357
------------------------------
                   Name Source
18              Aboleth  MM'14
19              Aboleth  MM'25
21      Abominable Yeti  MM'14
22      Abominable Yeti  MM'25
31   Adult Black Dragon  MM'14
..                  ...    ...
260            Archmage  MM'14
261            Archmage  MM'25
275            Armanite   MPMM
276            Armanite    MTF
285          Ash Zombie   LMoP

[100 rows x 2 columns]


In order to decide which source to trust when the same creature appears multiple times, We ranked books by release order using the following reference lists:

1. https://www.enworld.org/threads/d-d-releases-by-year-complete-list-almost.188346/
2. https://dungeonsanddragonsfan.com/dnd-5e-books-list/
3. https://dungeonsdragons.fandom.com/wiki/List_of_Dungeons_%26_Dragons_books

I grouped them by the year they were released and based their priority on how recent they are

In [10]:
# Define source priority
priority_map = {
    "MM'25": 100, "PHB'24": 100, "DMG'24": 100,
    'VEoR': 90, 'QftIS': 90, 'BMT': 90, 'MPP': 90, 'BGG': 90, 'PaBTSO': 90,
    'MPMM': 80, 'FTD': 80, 'DSotDQ': 80, 'DoSI': 80, 'JttRC': 80, 'LoX': 80, 'BAM': 80,
    'TCE': 70, 'IDRotF': 70, 'MOT': 70, 'ERLW': 70, 'GGR': 70, 'BGDIA': 70, 'GoS': 70,
    'VGM': 60, 'MTF': 60, 'XGE': 60, 'ToA': 60, 'TftYP': 60,
    "MM'14": 50, "PHB'14": 50, "DMG'14": 50, 'HotDQ': 50, 'RoT': 50, 'PotA': 50, 'CoS': 50
}

# Assign priority score to rows
df['Priority'] = df['Source'].map(priority_map).fillna(30)

# Sort by name and priority
df = df.sort_values(by=['Name', 'Priority'], ascending=[True, False])

# Identify duplicates
duplicate_mask = df.duplicated(subset=['Name'], keep=False)
dupes = df[duplicate_mask].copy()

# Get newest version of each duplicate
newest = dupes.drop_duplicates(subset=['Name'], keep='first')[['Name', 'Source']]
newest = newest.rename(columns={'Source': 'Newest_Source'})

# Get older versions of each duplicate
older = dupes[dupes.duplicated(subset=['Name'], keep='first')]
older_grouped = older.groupby('Name')['Source'].apply(lambda x: ', '.join(x)).reset_index()
older_grouped = older_grouped.rename(columns={'Source': 'Older_Source(s)'})

# Create comparison table
comparison_table = pd.merge(newest, older_grouped, on='Name')

# Print results
print("DUPLICATE COMPARISON TABLE")
print("-" * 80)
pd.set_option('display.max_rows', 100)
print(comparison_table.head(50))

DUPLICATE COMPARISON TABLE
--------------------------------------------------------------------------------
                      Name Newest_Source Older_Source(s)
0                  Aboleth         MM'25           MM'14
1          Abominable Yeti         MM'25           MM'14
2       Adult Black Dragon         MM'25           MM'14
3        Adult Blue Dragon         MM'25           MM'14
4       Adult Brass Dragon         MM'25           MM'14
5      Adult Bronze Dragon         MM'25           MM'14
6      Adult Copper Dragon         MM'25           MM'14
7        Adult Deep Dragon           FTD           FRAiF
8        Adult Gold Dragon         MM'25           MM'14
9       Adult Green Dragon         MM'25           MM'14
10           Adult Kruthik          MPMM             MTF
11             Adult Oblex          MPMM             MTF
12        Adult Red Dragon         MM'25           MM'14
13   Adult Sapphire Dragon           FTD            SADS
14     Adult Silver Dragon         MM

We look for any duplicates the compare their sources or books. The one with the higher source priority will be the one we will keep. Drop the source with the lower source priority

In [11]:
# Remove older duplicates
duplicates_removed = df.duplicated(subset=['Name']).sum()
df = df.drop_duplicates(subset=['Name'], keep='first')
df = df.drop(columns=['Priority'])

# Print summary
print(f"\nSuccessfully kept newest sources. {duplicates_removed} older versions removed.")


Successfully kept newest sources. 693 older versions removed.


In [12]:
# Check for remaining duplicates
duplicate_mask = df.duplicated(subset=['Name'], keep=False)
duplicates_df = df[duplicate_mask][['Name', 'Source']].sort_values(by='Name')

# Print summary
print(f"Total duplicate rows: {len(duplicates_df)}")
print("-" * 30)
print(duplicates_df.head(100))

Total duplicate rows: 0
------------------------------
Empty DataFrame
Columns: [Name, Source]
Index: []


In [13]:
# Now, let's see the columns
df[['Name', 'Source']]

,Name,Source
0,"""The Demogorgon""",IMR
1,Aarakocra,MM'14
2,Aarakocra Aeromancer,MM'25
3,Aarakocra Simulacrum,SKT
4,Aarakocra Skirmisher,MM'25
...,...,...
4448,Zress Orlezziir,WDMM
4449,Zuggtmoy,MPMM
4451,Zuleika Toranescu,CoS
4452,Zygfrek Belview,CoS


Now, that there are no more duplicates, let's continue cleaning

### Cleaning up HP values

We want HP to be a single number (not a formula like "2d8 + 4"), so we’ll extract the average/printed value and keep the underlying dice formula for reference.

In [14]:
# Check unique values and their frequency for the HP column
hp_counts = df['HP'].value_counts()

# Print the top 20 most common HP formats
print(f"Total Unique HP entries: {len(hp_counts)}")
print("-" * 30)
print(hp_counts.head(20))

Total Unique HP entries: 746
------------------------------
HP
9 (2d8)            71
11 (2d8 + 2)       68
27 (5d8 + 5)       67
33 (6d8 + 6)       59
40 (9d8)           53
52 (8d8 + 16)      53
4 (1d8)            49
78 (12d8 + 24)     49
58 (9d8 + 18)      48
16 (3d8 + 3)       45
22 (4d8 + 4)       44
65 (10d8 + 20)     44
22 (5d8)           38
112 (15d8 + 45)    37
66 (12d8 + 12)     36
27 (6d8)           35
1 (1d4 - 1)        34
45 (6d8 + 18)      34
67 (9d8 + 27)      33
44 (8d8 + 8)       32
Name: count, dtype: int64


For this we used regex. Regex: Starting at the left hand side, grab every number until we hit a letter or a space.

Look for invalid HP if there is one

In [ ]:
# Extract leading numbers from HP
extracted_hp = df['HP'].str.extract(r'^(\d+)')

# Find rows where extraction failed
failed_rows = df[extracted_hp[0].isna()]

# Print results
print(f"Rows failing conversion: {len(failed_rows)}")
print("-" * 30)
print(failed_rows[['Name', 'HP', 'CR']].head(20))



Rows failing conversion: 1
------------------------------
          Name                   HP                CR
2552  Meeseeks  —(immune to damage)  0 (XP 10; PB +2)


We used regex again but this time we also now wanna extract the hp formula. Regex: Find an open parenthesis and then grab everything that's inside until you bump into a closing parenthesis

In [16]:
# Extract dice formula from parentheses
df['HP_Formula'] = df['HP'].str.extract(r'\((.*?)\)').fillna('Static')

# Extract numeric average HP
df['HP_Numeric'] = df['HP'].str.extract(r'^(\d+)').astype(float)

# Drop rows with missing numeric values
df = df.dropna(subset=['HP_Numeric']).copy()

# Set HP as integer and remove temporary column
df['HP'] = df['HP_Numeric'].astype(int)
df = df.drop(columns=['HP_Numeric'])


print(df[['Name', 'HP', 'HP_Formula']].head(10))
df.info()

                    Name   HP  HP_Formula
0       "The Demogorgon"  123  13d12 + 39
1              Aarakocra   13         3d8
2   Aarakocra Aeromancer   66   12d8 + 12
3   Aarakocra Simulacrum    6         3d4
4   Aarakocra Skirmisher   11     2d8 + 2
5  Aarakocra Spelljammer   40         9d8
6           Aartuk Elder   75  10d10 + 20
7      Aartuk Starhorror   52    8d8 + 16
8        Aartuk Weedling   38     7d8 + 7
9       Aberrant Cultist  137   25d8 + 25
<class 'pandas.DataFrame'>
Index: 3658 entries, 0 to 4453
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Name                    3658 non-null   str  
 1   Source                  3658 non-null   str  
 2   Page                    3658 non-null   str  
 3   Size                    3658 non-null   str  
 4   Type                    3658 non-null   str  
 5   Alignment               3658 non-null   str  
 6   AC                      3658 non-n

We also now have dropped all invalid HP, all non numerical ones

## Cleaning up armor class (AC)

AC is usually a single number, but the raw data sometimes includes extra text. We'll extract just the numeric part.

In [17]:
# Check unique values and their frequency for the HP column
ac_counts = df['AC'].value_counts()

# Print the top 20 most common HP formats
print(f"Total Unique AC entries: {len(ac_counts)}")
print("-" * 30)
print(ac_counts.head(20))

Total Unique AC entries: 234
------------------------------
AC
12                         292
17 (natural armor)         229
13                         218
15 (natural armor)         213
16 (natural armor)         170
14 (natural armor)         150
10                         140
12 (15 with mage armor)    124
11                         124
14                         122
13 (natural armor)         109
18 (natural armor)         104
15                         103
19 (natural armor)          84
16                          74
17                          67
18 (plate armor)            63
15 (studded leather)        56
20 (natural armor)          55
13 (leather armor)          55
Name: count, dtype: int64


Much like the HP, we look for any invalid AC that we can drop later using regex

In [18]:
# Extract leading numbers from AC
extracted_ac = df['AC'].str.extract(r'^(\d+)')

# Find rows where extraction failed
failed_rows = df[extracted_ac[0].isna()]

# Print results
print(f"Rows failing conversion: {len(failed_rows)}")
print("-" * 30)
print(failed_rows[['Name', 'AC', 'CR']].head(20))

Rows failing conversion: 0
------------------------------
Empty DataFrame
Columns: [Name, AC, CR]
Index: []


It looks like there are none. So, let's jump straight to extraction

In [19]:
# Extract leading digits from AC
df['AC'] = df['AC'].str.extract(r'^(\d+)').astype(float)

# Convert AC to integer
df['AC'] = df['AC'].astype(int)

# Display result
df.info()

<class 'pandas.DataFrame'>
Index: 3658 entries, 0 to 4453
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Name                    3658 non-null   str  
 1   Source                  3658 non-null   str  
 2   Page                    3658 non-null   str  
 3   Size                    3658 non-null   str  
 4   Type                    3658 non-null   str  
 5   Alignment               3658 non-null   str  
 6   AC                      3658 non-null   int64
 7   HP                      3658 non-null   int64
 8   Speed                   3658 non-null   str  
 9   Strength                3658 non-null   str  
 10  Dexterity               3658 non-null   str  
 11  Constitution            3658 non-null   str  
 12  Intelligence            3658 non-null   str  
 13  Wisdom                  3658 non-null   str  
 14  Charisma                3658 non-null   str  
 15  Saving Throws           3658 non-null

We extract only the first integer seen from left to right and changed it's datatype to int

## Ability scores (STR/DEX/CON/etc.)

These should all be integers, but it's common for scraped sources to include text or weird formatting. We'll verify and clean them.

In [20]:
# List ability score columns
ability_cols = ['Strength', 'Dexterity', 'Constitution', 'Intelligence', 'Wisdom', 'Charisma']

# Calculate and print frequency for each column
for col in ability_cols:
    ability_counts = df[col].value_counts()
    
    print(f"REPORT: {col.upper()}")
    print("-" * 30)
    print(f"Total Unique Entries: {len(ability_counts)}")
    print("Top 10 frequent values:")
    print(ability_counts.head(10))
    print("\n")

# Check for non-numeric strings
non_numeric_found = False
for col in ability_cols:
    invalid_entries = df[~df[col].astype(str).str.isnumeric()][col].unique()
    if len(invalid_entries) > 0:
        print(f"Warning: Non-numeric values in {col}: {invalid_entries}")
        non_numeric_found = True

# Print verification result
if not non_numeric_found:
    print("Verification Result: All ability scores are numeric.")

REPORT: STRENGTH
------------------------------
Total Unique Entries: 30
Top 10 frequent values:
Strength
10    405
18    354
16    303
11    257
14    208
15    206
12    192
19    172
20    151
13    151
Name: count, dtype: int64


REPORT: DEXTERITY
------------------------------
Total Unique Entries: 26
Top 10 frequent values:
Dexterity
14    710
10    478
16    395
12    392
15    358
13    250
11    245
18    199
17    134
9     111
Name: count, dtype: int64


REPORT: CONSTITUTION
------------------------------
Total Unique Entries: 25
Top 10 frequent values:
Constitution
14    493
12    470
16    440
10    319
13    278
18    270
15    238
11    217
17    198
20    193
Name: count, dtype: int64


REPORT: INTELLIGENCE
------------------------------
Total Unique Entries: 29
Top 10 frequent values:
Intelligence
10    541
11    299
12    291
2     234
14    217
3     201
13    186
16    176
17    170
6     167
Name: count, dtype: int64


REPORT: WISDOM
------------------------------


In [21]:
# List ability score columns
ability_cols = ['Strength', 'Dexterity', 'Constitution', 'Intelligence', 'Wisdom', 'Charisma']

# Convert columns to integers
df[ability_cols] = df[ability_cols].astype(int)

# Print conversion results
print("CONVERSION REPORT")
print("-" * 30)
print(f"Rows processed: {len(df)}")
print(f"Target columns: {ability_cols}")
print(f"New datatypes:\n{df[ability_cols].dtypes}")

# Check for uniform integer types
is_uniform = all(df[col].apply(type).unique()[0] == int for col in ability_cols)
print(f"Check Passed: {is_uniform}")

# Display structure
df.info()

CONVERSION REPORT
------------------------------
Rows processed: 3658
Target columns: ['Strength', 'Dexterity', 'Constitution', 'Intelligence', 'Wisdom', 'Charisma']
New datatypes:
Strength        int64
Dexterity       int64
Constitution    int64
Intelligence    int64
Wisdom          int64
Charisma        int64
dtype: object
Check Passed: True
<class 'pandas.DataFrame'>
Index: 3658 entries, 0 to 4453
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Name                    3658 non-null   str  
 1   Source                  3658 non-null   str  
 2   Page                    3658 non-null   str  
 3   Size                    3658 non-null   str  
 4   Type                    3658 non-null   str  
 5   Alignment               3658 non-null   str  
 6   AC                      3658 non-null   int64
 7   HP                      3658 non-null   int64
 8   Speed                   3658 non-null   str  
 9

We just converted all the ability scores (str, dex, etc) all in to integers

## Parsing challenge rating (CR)

CR can be a number or a fraction (e.g., "1/2"). We'll normalize it into a float so it can be used in models.

In [22]:
# Extract numeric or fraction string from CR
df['CR'] = df['CR'].str.extract(r'^([0-9/]+)')

# Convert fraction strings to decimals
def convert_fraction(val):
    if pd.isna(val): return None
    if '/' in val:
        num, den = val.split('/')
        return float(num) / float(den)
    return float(val)

df['CR'] = df['CR'].apply(convert_fraction)

# Drop rows without a CR
df = df.dropna(subset=['CR']).copy()

# Print verification results
print(f"Dataset Size: {len(df)} rows")
print(f"Column Type:  {df['CR'].dtype}")
print(f"Unique CRs:   {sorted(df['CR'].unique())}")

# Display structure
df.info()

Dataset Size: 3658 rows
Column Type:  float64
Unique CRs:   [np.float64(0.0), np.float64(0.125), np.float64(0.25), np.float64(0.5), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(30.0)]
<class 'pandas.DataFrame'>
Index: 3658 entries, 0 to 4453
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    3658 non-null   str    
 1   Source                  3658 non-null   str    
 2   Page                    3658 non-null   str    

The values of CR has factions on it. We just converted evrything to float so that every value has the same datatype (float)

## Normalizing text fields

To make comparisons reliable, we lowercase text fields, trim whitespace, and remove stray quotes.

In [23]:
# Identify string columns
string_cols = df.select_dtypes(include=['object']).columns

# Normalize text data
for col in string_cols:
    # Lowercase and strip whitespace
    df[col] = df[col].astype(str).str.lower().str.strip()
    
    # Remove quotation marks
    df[col] = df[col].str.replace('"', '', regex=False).str.replace("'", "", regex=False)


display(df.head())

C:\Users\goaar\AppData\Local\Temp\ipykernel_12488\1847918084.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include=['object']).columns


,Name,Source,Page,Size,Type,Alignment,AC,HP,Speed,Strength,...,Actions,Bonus Actions,Reactions,Legendary Actions,Mythic Actions,Lair Actions,Regional Effects,Environment,Treasure,HP_Formula
0,the demogorgon,imr,53,large,giant,chaotic neutral,15,123,40 ft.,21,...,multiattack. the ettin makes two attacks: one ...,none,none,none,none,none,none,unknown,none,13d12 + 39
1,aarakocra,mm14,12,medium,humanoid (aarakocra),neutral good,12,13,"20 ft., fly 50 ft.",10,...,"talon. melee weapon attack: +4 to hit, reach 5...",none,none,none,none,none,none,mountain,none,3d8
2,aarakocra aeromancer,mm25,10,medium,elemental,neutral,16,66,"20 ft., fly 50 ft.",10,...,multiattack. the aarakocra makes two wind staf...,none,feather fall (1/day). the aarakocra casts feat...,none,none,none,none,"mountain, planar (elemental plane of air)","individual, implements",12d8 + 12
3,aarakocra simulacrum,skt,188,medium,humanoid (aarakocra),neutral good,12,6,"20 ft., fly 50 ft.",10,...,"talon. melee weapon attack: +4 to hit, reach 5...",none,none,none,none,none,none,unknown,none,3d4
4,aarakocra skirmisher,mm25,10,medium,elemental,neutral,12,11,"20 ft., fly 50 ft.",10,...,"talons. melee attack roll: +4, reach 5 ft. hit...",none,none,none,none,none,none,"mountain, planar (elemental plane of air)","individual, implements",2d8 + 2


### Extracting values and enforcing types

For this one, the regex will split the text into two groups. The text before the parenthesis and the texts inside it

In [24]:
# Extract base type and subtype
extracted = df['Type'].str.extract(r'([^(]+)\(([^)]+)\)')

# Update Type with base text
df['Type'] = extracted[0].str.strip().fillna(df['Type'])

# Create Subtype column
df['Subtype'] = extracted[1].str.strip()

# Fill missing subtypes
df['Subtype'] = df['Subtype'].fillna('none')


display(df[['Name', 'Type', 'Subtype']].head(10))

,Name,Type,Subtype
0,the demogorgon,giant,none
1,aarakocra,humanoid,aarakocra
2,aarakocra aeromancer,elemental,none
3,aarakocra simulacrum,humanoid,aarakocra
4,aarakocra skirmisher,elemental,none
5,aarakocra spelljammer,humanoid,aarakocra
6,aartuk elder,plant,none
7,aartuk starhorror,plant,none
8,aartuk weedling,plant,none
9,aberrant cultist,humanoid,none


In [25]:
df.info()

<class 'pandas.DataFrame'>
Index: 3658 entries, 0 to 4453
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    3658 non-null   str    
 1   Source                  3658 non-null   str    
 2   Page                    3658 non-null   str    
 3   Size                    3658 non-null   str    
 4   Type                    3658 non-null   str    
 5   Alignment               3658 non-null   str    
 6   AC                      3658 non-null   int64  
 7   HP                      3658 non-null   int64  
 8   Speed                   3658 non-null   str    
 9   Strength                3658 non-null   int64  
 10  Dexterity               3658 non-null   int64  
 11  Constitution            3658 non-null   int64  
 12  Intelligence            3658 non-null   int64  
 13  Wisdom                  3658 non-null   int64  
 14  Charisma                3658 non-null   int64  
 15  Sav

In [26]:
# Save prepared dataset to CSV
prepared_df = df.copy()
prepared_df.to_csv('../data/prepared_bestiary.csv', index=False)

# Print summary and info
print("Dataset saved.")
prepared_df.info()

Dataset saved.
<class 'pandas.DataFrame'>
Index: 3658 entries, 0 to 4453
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    3658 non-null   str    
 1   Source                  3658 non-null   str    
 2   Page                    3658 non-null   str    
 3   Size                    3658 non-null   str    
 4   Type                    3658 non-null   str    
 5   Alignment               3658 non-null   str    
 6   AC                      3658 non-null   int64  
 7   HP                      3658 non-null   int64  
 8   Speed                   3658 non-null   str    
 9   Strength                3658 non-null   int64  
 10  Dexterity               3658 non-null   int64  
 11  Constitution            3658 non-null   int64  
 12  Intelligence            3658 non-null   int64  
 13  Wisdom                  3658 non-null   int64  
 14  Charisma                3658 non-null   i

Summary of Data Preparation Decisions
1. We dropped all rows with missing or "Unknown" Challenge Ratings (CR) because CR is the primary target for our models and keeping rows without it would add noise.

2. We created a Source Priority Tier (100, 90, 80) so that newer 2024/2025 versions of monsters automatically overwrite older 2014 data during duplicate removal.

3. We filled missing text fields like Skills or Resistances with "None" and missing environments with "Unknown" to preserve dataset size while making features readable for the models.

4. We used Regex to extract leading numbers from HP and AC strings into a clean numerical format while saving the dice formulas in a separate column for reference.

5. We used Regex to split monster types into two columns for Base Type and Subtype so the models can learn both broad and specific patterns.

6. We converted all Challenge Rating fractions into decimals and forced all text fields to lowercase to ensure consistent grouping and data types.

7. We cast all ability scores and combat statistics to integers or floats to prevent the neural network and classical models from crashing during training.